# Lens 2 — Notebook 06: Product-Demand Data Preparation & Feature Engineering
**IT3091 Machine Learning — Industry Explorer Track**  
**Group ID:** `2026-AI-36` | **Primary Author:** `IT24102913`  
**Lens:** 2 — Product-Demand Forecasting  
**Role:** Product-Demand Data Preparation & Feature Engineering

---

## Notebook Purpose
This notebook transforms the panel demand data (NB05) into a **machine-learning-ready feature matrix** for all 11 dishes simultaneously.

Key differences from Lens 1 (revenue) feature engineering:
1. **Panel structure**: rows are (date, item) pairs — requires item identity features
2. **Per-item lag features**: lag-k for item `i` on day `t` = quantity of item `i` on day `t-k`
3. **Cross-item features**: total demand across all items can signal overall trading volume
4. **Item one-hot encoding**: allows the model to learn item-specific baselines

---

## Section Overview
1. Environment Setup
2. Load Panel Data
3. Continuous Calendar Reindexing
4. Calendar & Cyclical Features
5. Per-Item Autoregressive Lag Features
6. Rolling Window Statistics (Leakage-Free)
7. Cross-Item Features
8. Item Identity Features (One-Hot Encoding)
9. Feature Matrix Assembly & Validation
10. Leakage Verification
11. Save Feature Matrix
12. Feature Engineering Decision Log

---
## 1. Environment Setup

In [1]:
import os, sys, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print("Environment initialised.")

Environment initialised.


---
## 2. Load Panel Data

In [2]:
PANEL_PATH = '../data/processed/daily_product_demand_long.csv'
panel_df = pd.read_csv(PANEL_PATH, parse_dates=['Date'])
panel_df = panel_df.sort_values(['ItemName', 'Date']).reset_index(drop=True)

CORE_ITEMS = sorted(panel_df['ItemName'].unique().tolist())

print(f"Panel shape: {panel_df.shape}")
print(f"Items: {CORE_ITEMS}")
print(f"Date range: {panel_df['Date'].min().date()} → {panel_df['Date'].max().date()}")
print()
print(panel_df.head())

Panel shape: (9845, 3)
Items: ['BREAKFAST', 'BUTTER CAKE', 'CHICKEN CURRY', 'CHICKEN RICE FULL', 'DATE CAKE', 'LUNCH', 'PATIS', 'ROLLS', 'SAMOSA', 'TEA', 'WATALAPPAN']
Date range: 2024-03-27 → 2026-09-07

        Date   ItemName   Qty
0 2024-03-27  BREAKFAST   4.0
1 2024-03-28  BREAKFAST   5.0
2 2024-03-29  BREAKFAST  12.0
3 2024-03-30  BREAKFAST  20.0
4 2024-03-31  BREAKFAST   7.0


---
## 3. Continuous Calendar Reindexing

Verify the panel already covers a continuous calendar (created in NB05). Each item must have exactly one row per day.

In [3]:
n_days = (panel_df['Date'].max() - panel_df['Date'].min()).days + 1
n_items = panel_df['ItemName'].nunique()
expected_rows = n_days * n_items

print(f"Calendar days in span : {n_days}")
print(f"Items                 : {n_items}")
print(f"Expected rows         : {expected_rows:,}")
print(f"Actual rows           : {len(panel_df):,}")
print(f"Missing rows          : {expected_rows - len(panel_df)}")
assert len(panel_df) == expected_rows, "Panel is not balanced! Check NB05 output."
print("✅ Panel is balanced — one row per (date, item) pair.")

Calendar days in span : 895
Items                 : 11
Expected rows         : 9,845
Actual rows           : 9,845
Missing rows          : 0
✅ Panel is balanced — one row per (date, item) pair.


---
## 4. Calendar & Cyclical Features

Same calendar feature set as Lens 1, applied to each row in the panel.

In [4]:
df = panel_df.copy()

# Cyclical encodings
df['dow_sin']        = np.sin(2 * np.pi * df['Date'].dt.dayofweek / 7)
df['dow_cos']        = np.cos(2 * np.pi * df['Date'].dt.dayofweek / 7)
df['month_sin']      = np.sin(2 * np.pi * df['Date'].dt.month / 12)
df['month_cos']      = np.cos(2 * np.pi * df['Date'].dt.month / 12)
df['dom_sin']        = np.sin(2 * np.pi * df['Date'].dt.day / 31)
df['dom_cos']        = np.cos(2 * np.pi * df['Date'].dt.day / 31)

# Binary indicators
df['is_weekend']     = (df['Date'].dt.dayofweek >= 5).astype(int)
df['is_friday']      = (df['Date'].dt.dayofweek == 4).astype(int)
df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
df['is_month_end']   = df['Date'].dt.is_month_end.astype(int)
df['quarter']        = df['Date'].dt.quarter
df['t_index']        = df.groupby('ItemName').cumcount()  # per-item time index

calendar_features = [
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'dom_sin', 'dom_cos', 'is_weekend', 'is_friday',
    'is_month_start', 'is_month_end', 'quarter', 't_index'
]

print(f"Calendar features: {len(calendar_features)}")
print(calendar_features)

Calendar features: 12
['dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'dom_sin', 'dom_cos', 'is_weekend', 'is_friday', 'is_month_start', 'is_month_end', 'quarter', 't_index']


---
## 5. Per-Item Autoregressive Lag Features

**Critical design note:** Lag features must be computed *within each item* (grouped by ItemName).  
Lag-1 for LUNCH on Tuesday = LUNCH quantity on Monday (NOT BREAKFAST quantity on Monday).

In [5]:
LAG_DAYS = [1, 2, 3, 7, 14, 21]

# Sort by item then date before computing lags
df = df.sort_values(['ItemName', 'Date']).reset_index(drop=True)

for k in LAG_DAYS:
    df[f'lag_{k}'] = df.groupby('ItemName')['Qty'].shift(k)

lag_features = [f'lag_{k}' for k in LAG_DAYS]
print(f"Per-item lag features created: {lag_features}")
print(f"Rows with NaN lags (warm-up period): {df[lag_features].isna().any(axis=1).sum():,}")
print(f"Expected: {len(CORE_ITEMS)} items × 21 max_lag = {len(CORE_ITEMS)*21} rows")

Per-item lag features created: ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_21']
Rows with NaN lags (warm-up period): 231
Expected: 11 items × 21 max_lag = 231 rows


---
## 6. Rolling Window Statistics (Leakage-Free)

Per-item rolling statistics, computed on shifted series to prevent target leakage.

In [6]:
WINDOWS = [7, 14]

df = df.sort_values(['ItemName', 'Date']).reset_index(drop=True)

for w in WINDOWS:
    shifted = df.groupby('ItemName')['Qty'].shift(1)
    df[f'roll{w}_mean'] = df.groupby('ItemName')['Qty'].shift(1).transform(
        lambda x: x.rolling(w, min_periods=1).mean()
    )
    df[f'roll{w}_std'] = df.groupby('ItemName')['Qty'].shift(1).transform(
        lambda x: x.rolling(w, min_periods=2).std()
    )
    df[f'roll{w}_max'] = df.groupby('ItemName')['Qty'].shift(1).transform(
        lambda x: x.rolling(w, min_periods=1).max()
    )

rolling_features = (
    [f'roll{w}_mean' for w in WINDOWS] +
    [f'roll{w}_std'  for w in WINDOWS] +
    [f'roll{w}_max'  for w in WINDOWS]
)

print(f"Rolling features created: {len(rolling_features)}")
print(rolling_features)

Rolling features created: 6
['roll7_mean', 'roll14_mean', 'roll7_std', 'roll14_std', 'roll7_max', 'roll14_max']


---
## 7. Cross-Item Features

The total demand across all items on the same day reflects the overall restaurant trading volume. A busy restaurant day for LUNCH likely coincides with a busy day for TEA too.

In [7]:
# Total quantity across all items per day
daily_total = df.groupby('Date')['Qty'].sum().rename('daily_total_qty')
df = df.merge(daily_total, on='Date', how='left')

# Lagged total (shift by 1 to prevent leakage — we use yesterday's total as context)
date_total_lag1 = (
    df.drop_duplicates('Date')
    .set_index('Date')['daily_total_qty']
    .shift(1)  # previous day's total
    .rename('total_qty_lag1')
)
df = df.merge(date_total_lag1.reset_index(), on='Date', how='left')

# Remove the raw daily_total (it includes current day = leakage)
df = df.drop(columns=['daily_total_qty'])

cross_features = ['total_qty_lag1']
print(f"Cross-item features: {cross_features}")

Cross-item features: ['total_qty_lag1']


---
## 8. Item Identity Features (One-Hot Encoding)

One-hot encoding allows each item to have its own intercept (baseline demand level) in the model. Without this, the model would apply the same baseline to all dishes.

In [8]:
ohe_df = pd.get_dummies(df['ItemName'], prefix='item', drop_first=False).astype(int)
df = pd.concat([df, ohe_df], axis=1)

item_features = ohe_df.columns.tolist()
print(f"Item one-hot features: {len(item_features)}")
print(item_features)

Item one-hot features: 11
['item_BREAKFAST', 'item_BUTTER CAKE', 'item_CHICKEN CURRY', 'item_CHICKEN RICE FULL', 'item_DATE CAKE', 'item_LUNCH', 'item_PATIS', 'item_ROLLS', 'item_SAMOSA', 'item_TEA', 'item_WATALAPPAN']


---
## 9. Feature Matrix Assembly & Validation

In [9]:
ALL_FEATURES = calendar_features + lag_features + rolling_features + cross_features + item_features

# Drop warm-up rows (NaN from lags)
df_model = df[['Date', 'ItemName', 'Qty'] + ALL_FEATURES].dropna().reset_index(drop=True)

print(f"Feature matrix shape : {df_model.shape}")
print(f"Features             : {len(ALL_FEATURES)}")
print(f"Target               : Qty")
print(f"Observations         : {len(df_model):,}")
print(f"Date range           : {df_model['Date'].min().date()} → {df_model['Date'].max().date()}")
print(f"NaN after dropna     : {df_model[ALL_FEATURES].isna().sum().sum()}")  

Feature matrix shape : (9614, 39)
Features             : 36
Target               : Qty
Observations         : 9,614
Date range           : 2024-04-17 → 2026-09-07
NaN after dropna     : 0


In [10]:
print("Feature Group Summary:")
print(f"  Calendar/cyclical  : {len(calendar_features)}")
print(f"  Per-item lags      : {len(lag_features)}")
print(f"  Rolling statistics : {len(rolling_features)}")
print(f"  Cross-item         : {len(cross_features)}")
print(f"  Item identity (OHE): {len(item_features)}")
print(f"  TOTAL              : {len(ALL_FEATURES)}")

# Correlation of each group with target
corr = df_model[['Qty'] + ALL_FEATURES].corr()['Qty'].drop('Qty').abs().sort_values(ascending=False)
print("\nTop 15 Features by |Pearson r| with Qty:")
for feat, val in corr.head(15).items():
    print(f"  {feat:<25}: {val:.4f}")

Feature Group Summary:
  Calendar/cyclical  : 12
  Per-item lags      : 6
  Rolling statistics : 6
  Cross-item         : 1
  Item identity (OHE): 11
  TOTAL              : 36



Top 15 Features by |Pearson r| with Qty:


  roll7_mean               : 0.6507
  roll14_mean              : 0.6498
  lag_1                    : 0.5507
  roll7_max                : 0.5470
  roll14_max               : 0.5058
  lag_3                    : 0.5039
  lag_2                    : 0.4892
  roll14_std               : 0.4809
  roll7_std                : 0.4687
  lag_7                    : 0.4669
  lag_14                   : 0.4401
  lag_21                   : 0.4197
  item_LUNCH               : 0.3567
  item_BREAKFAST           : 0.2681
  item_ROLLS               : 0.2362


---
## 10. Leakage Verification

In [11]:
# Per-item future correlation check
df_check = df_model.copy()
df_check['Qty_next'] = df_check.groupby('ItemName')['Qty'].shift(-1)
df_check = df_check.dropna(subset=['Qty_next'])

leak_corr = df_check[ALL_FEATURES].corrwith(df_check['Qty_next']).abs().sort_values(ascending=False)

print("Top 10 features by correlation with NEXT DAY demand (leakage check):")
print(leak_corr.head(10).round(4))
print()
if leak_corr.max() > 0.99:
    print("⚠️  WARNING: Possible leakage detected!")
else:
    print(f"✅ Max correlation with next-day demand: {leak_corr.max():.4f} — No leakage detected.")

Top 10 features by correlation with NEXT DAY demand (leakage check):


roll14_mean    0.6377
roll7_mean     0.6318
roll7_max      0.5217
lag_2          0.5031
lag_3          0.4917
lag_1          0.4894
roll14_max     0.4869
roll14_std     0.4627
lag_7          0.4593
roll7_std      0.4458
dtype: float64

✅ Max correlation with next-day demand: 0.6377 — No leakage detected.


---
## 11. Save Feature Matrix

In [12]:
FEAT_OUT = '../data/processed/demand_feature_matrix.csv'
df_model.to_csv(FEAT_OUT, index=False)
print(f"Saved demand feature matrix: {FEAT_OUT}  ({df_model.shape})")

# Save metadata
meta = {
    'feature_cols': ALL_FEATURES,
    'target_col': 'Qty',
    'date_col': 'Date',
    'item_col': 'ItemName',
    'items': CORE_ITEMS,
    'n_rows': len(df_model),
    'date_min': str(df_model['Date'].min().date()),
    'date_max': str(df_model['Date'].max().date()),
    'author': 'IT24102913'
}
META_OUT = '../data/processed/demand_feature_meta.json'
with open(META_OUT, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"Saved metadata: {META_OUT}")

Saved demand feature matrix: ../data/processed/demand_feature_matrix.csv  ((9614, 39))
Saved metadata: ../data/processed/demand_feature_meta.json


---
## 12. Feature Engineering Decision Log

| Decision | Rationale | Alternative Considered | Why Rejected |
|---|---|---|---|
| Per-item lag grouping | Lag-1 for LUNCH must be LUNCH's previous day — mixing items is semantically wrong | Global lag (ignoring item identity) | Would average across dishes — LUNCH lag≠TEA lag |
| One-hot encoding for items | Allows separate baselines per item | Target encoding (mean demand per item) | Target encoding leaks aggregate demand into features |
| Cross-item lag1 (total_qty_lag1) | Yesterday's total restaurant traffic signals today's expected volume | Current day total (no lag) | Using today's total = leakage |
| Drop first 21 rows per item | Max lag = 21; warmup period must be discarded | Fill NaN with 0 | Zero imputation introduces false signal on warmup rows |
| Rolling windows grouped per item | Each item has its own demand rhythm | Global rolling (across all items) | Averaging across items would dilute item-specific patterns |

### AI-use Declaration (IT24102913)
All feature engineering decisions, rationales, and code in this notebook are the work of the student author. Generative AI was used only for minor code style suggestions. The analytical design follows the course ML pipeline principles.